## Setup

Make sure that you install the requirements into a venv and enable that kernel before running this notebook.

In [407]:
import json
import pandas as pd
import numpy as np

from collections import Counter

## Data Ingestion

In [427]:
def format_raw_json():
    """Extract complete structure of stringified JSON files.
    
    Parses agentic policy records to view different metrics.
    
    Args:
        None
    
    Returns:
        pd.DataFrame: Returns complete DataFrame or one that is empty.
    """
    rows = []
    with open('./interactive_agent.jsonl', 'r') as agent_file:
        counter = 0
        for line in agent_file:
            # line comes in as a raw string, need loads()
            line = json.loads(line)

            # initialize Counters for roles and tool calls
            tool_counts = Counter()
            role_counts = Counter()

            # initialize containers for not blocking the DataFrame generation
            total_tool_calls = 0
            system_prompt = ""
            num_turns = 0
            num_user = 0
            num_assistant = 0
            num_tool = 0
            num_system = 0
            tools_called = 0
            num_tool_calls = 0
            num_tools_available = 0
            user_first_message = ""
            messages = []
            domain = ""
            license = ""
            uuid = line.get("uuid")
            messages = line.get("messages")
            for message in messages:
                role = message.get("role")
                role_counts.update([role])
                if role == "system" and system_prompt == "":
                    system_prompt = message.get("content")
                if role == "user" and user_first_message == "":
                    user_first_message = message.get("content")
                if "tool_calls" in message:
                    tool_calls = message.get("tool_calls")
                    
                    # drop down to tool_calls and log their frequency
                    for call in tool_calls:
                        function = call.get("function")
                        name = function.get("name")
                        tool_counts.update([name])
                num_turns = len(messages)
                num_user = role_counts["user"]
                num_assistant = role_counts["assistant"]
                num_tool = role_counts["tool"]
                num_system = role_counts["system"]
                tools_called = tool_counts.keys()
                num_tool_calls = sum(tool_counts.values())
                license = line.get("license", np.nan)
                has_reasoning = "reasoning" in line

            # Build list of dicts
            rows.append({
                "uuid": uuid,
                "messages": messages, # Could be seen as excessive and a waste of space unless we are completely transitioning to a new dataset.
                "system_prompt": system_prompt,
                "num_turns": num_turns,
                "num_user": num_user,
                "num_assistant": num_assistant,
                "num_tool": num_tool,
                "num_system": num_system,
                "tools_called": tools_called,
                "num_tool_calls": num_tool_calls,
                "user_first_message": user_first_message,
                "license": license,
                "has_reasoning": has_reasoning
            })
    return pd.DataFrame(rows)
df = format_raw_json()

### Ingestion Function
format_raw_json() is core to parsing this document since our dataset is actually a collection of json strings!  There was a larger plan of formatting all of the agent policies into dicts, but eventually I realized that it didn't matter as much as I thought it would.  Mainly because we're cleaning data and I don't have the inference engine of AI to do that part justice.  However, we were able to count frequencies of user's message count, the amount of tools call, and finding it all reliably.  In order for this to be a pandas DataFrame we must use this function, or a version of it, to produce the dataset we analyze later on.

 0   uuid                19028 non-null  object
 1   messages            19028 non-null  object
 2   system_prompt       19028 non-null  object
 3   num_turns           19028 non-null  int64 
 4   num_user            19028 non-null  int64 
 5   num_assistant       19028 non-null  int64 
 6   num_tool            19028 non-null  int64 
 7   num_system          19028 non-null  int64 
 8   tools_called        19028 non-null  object
 9   num_tool_calls      19028 non-null  int64 
 10  user_first_message  19028 non-null  object
 11  license             19028 non-null  object
 12  has_reasoning       19028 non-null  bool  

* uuid: is important to track individual instances
* messages: formats the agent's policy and actions in a neat list 
* system_prompt: helps preserve the original policy raw
* num_turns: preserves the number of total interactions (user & agent messages/actions)
* num_user: preserves the total number of user initiated messages
* num_assistant: preserves the total number of agent initiated messages
* num_tool: preserves the total number of tool calls initiated
* num_system: this field is for validation, there should only be 1 system message in each policy
* tools_called: a tuple that preserves the list of tool calls from the agent
* num_tool_calls: total number of tool calls to support filtering the DataFrame
* user_first_message: included just to demonstrate important data and signal series for training
* license: important for legal details and any additional analysis for more diverse datasets
* has_reasoning: flags whether the policy has reasoning, good for training data

#### Data Details
There aren't any floats in this dataset and all numbers are integers or objects.

Has reasoning signals whether this agent is using ReACT or not.